# 1.0 Import libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# 2.0 Data load

In [3]:
gas_average = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/gas_clean_average_subsets_1234561h_2026_09_26.csv')
nutrition_average = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/nutrition_average_1234561h_2026_09_26.csv')
grasses_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/grasses_categories.csv')
legumes_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/legumes_categories.csv')
shrubs_categories = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_09_29_gas_nutrition_and_categories_compilation/shrubs_categories.csv')

# 3.0 Gas and nutrition data merge

In [4]:
gas_average.columns

Index(['id_lab', 'id', 'subset', 'no', 'requisitioner', 'tax_name',
       'functional_group', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [5]:
gas_average_2 = gas_average.drop(['id', 'subset', 'no', 'requisitioner', 'tax_name', 'functional_group'], axis=1)

In [6]:
gas_nutrition = pd.merge(nutrition_average, gas_average_2, on="id_lab", how="outer")

# 4.0 Calculating quartiles

In [7]:
def rank_and_quartiles(df):
    # Sort by tddm (descending) and ch4_intensity (ascending)
    df = df.sort_values(by=["TDDM", "CH4_Intensity"], ascending=[False, True]).reset_index(drop=True)
    # Add ascending ranking
    df.insert(0, "quartile_rank", range(1, len(df) + 1))
    # Assign quartiles based on ranking
    df["quartile"] = pd.qcut(df["quartile_rank"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
    return df

In [8]:
grasses_categories_2 = rank_and_quartiles(grasses_categories)
legumes_categories_2 = rank_and_quartiles(legumes_categories)
shrubs_categories_2 = rank_and_quartiles(shrubs_categories)

# 5.0 Adding functional group categories to the dataframe

In [9]:
# Extract useful columns
grasses_categories_3 = grasses_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_category', 'Overall_Rank', 'quartile','quartile_rank']]
legumes_categories_3 = legumes_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_category', 'Overall_Rank', 'quartile','quartile_rank']]
shrubs_categories_3 = shrubs_categories_2[['Accession_ID', 'CH4_Category',  'TDDM_Category', 'lmf_category', 'Overall_Rank', 'quartile','quartile_rank']]

In [10]:
# Rename columns
grasses_categories_3 = grasses_categories_3.rename(columns={"Accession_ID": "id_lab", "Overall_Rank":"lmf_category_rank"})
legumes_categories_3 = legumes_categories_3.rename(columns={"Accession_ID": "id_lab",  "Overall_Rank":"lmf_category_rank"})
shrubs_categories_3 = shrubs_categories_3.rename(columns={"Accession_ID": "id_lab",  "Overall_Rank":"lmf_category_rank"})

# 6.0 Compiling data

In [11]:
categories = pd.concat([grasses_categories_3, legumes_categories_3, shrubs_categories_3], ignore_index=True)

In [12]:
gas_nutrition_categories = pd.merge(gas_nutrition, categories, on="id_lab", how="outer")

# 7.0 Curation

Manual curation on 2026_09_30 detected irregularities on samples F26-0087, F26-0042, F25-1772, and F25-1692.

Decisions:

F26-0087 must be delete it, Nutrition data is absent, so this accession will be included next time.

F26-0042 does not contains gas data, but its duplication "F26-0042-" contains gas data but the nutrition data is not correct. So nutrition data from "F26-0042" will be placed on "F26-0042-" and "F26-0042-" will be renamed to "F26-0042".

F25-1772 does not contain gas data, but its duplication "F25-1772-" contains both nutrition and gas data, the nutrition data is the same, so the accession F25-1772 will be delete and the "F25-1772-" will be renamed to "F25-1772".

F25-1692 does not contain gas data, but its duplication "F25-1692-" contains both nutrition and gas data, the nutrition data is the same, so the accession F25-1692 will be delete and the "F25-1692-" will be renamed to "F25-1772".

F24-3545 In a discussion with Johanna, we decided delete this sample because at lab analysis the amount of sample was not enough for the gas production analysis.

Note. The next samples are from other centers so must be delete them from this database:

F25-0798, F25-0814, F25-0818, F25-0819, F25-0845, F25-0848,  F25-0863, F25-0867, F25-0869, F25-0873, F25-0874, F25-0878.

These accessions will be deleted here, but they must be deleted during SQL process for dataframe creation next time.



############ IMPORTANT#############


F25-0883 F25-0891, F25-0893, F25-0897, F25-0899, F25-0907, F25-0923, F25-0924  are from ciat but do not have nutrition data.

Only F25-0907, F25-0883 where categorized. The others are outside of this categorization. **THEY MUST BE INCLUDED ON NEXT TIME**. I have to revie why the stay out.






In [13]:
original_id = "F26-0042"
duplicate_id = "F26-0042-"

nutrition_cols = ["dm_percentage", "ash_dm","om_percentage","pc_percentage_dm","adf_percentage_dm","ndf_percentage_dm","n_replicates_nutrition"]

nutrition_data = gas_nutrition.loc[gas_nutrition["id_lab"] == original_id, nutrition_cols]

if len(gas_nutrition_categories) == 1:
    # Replace nutrition data
    gas_nutrition.loc[gas_nutrition["id_lab"] == duplicate_id, nutrition_cols] = gas_nutrition_categories.iloc[0].values
else:
    print("Error: F26-0042 does not have exactly one nutrition record.")

Error: F26-0042 does not have exactly one nutrition record.


In [14]:
to_delete = ['F26-0042','F26-0087','F25-1772','F25-1692', 'F24-3545',    'F25-0883', 'F25-0891', 'F25-0893', 'F25-0897', 'F25-0899', 'F25-0907', 'F25-0923', 'F25-0924']

In [15]:
len(gas_nutrition)

1232

In [16]:
gas_nutrition_categories = gas_nutrition_categories[~gas_nutrition_categories['id_lab'].isin(to_delete)].copy()

In [17]:
len(gas_nutrition_categories)

1219

In [18]:
id_lab_changes = {'F26-0042-': 'F26-0042', 'F25-1772-': 'F25-1772', 'F25-1692-': 'F25-1692'}
gas_nutrition_categories['id_lab'] = gas_nutrition_categories['id_lab'].replace(id_lab_changes)

In [26]:
gas_nutrition_categories.to_csv('/content/drive/MyDrive/lmf/output/2026_09_29_gas_nutrition_and_categories_compilation/gas_nutrition_categories_2026_10_01.csv', index=None)

#7.0 Comparing project dataframes

In [20]:
gas_nutrition_categories_list = gas_nutrition_categories["id"].unique().tolist()
print('gas_nutrition_categories_list:', len(gas_nutrition_categories_list), "ids")

gas_average_list = gas_average["id"].unique().tolist()
print('gas_average:', len(gas_average_list), "ids")

nutrition_average_list = nutrition_average["id"].unique().tolist()
print('nutrition_average:', len(nutrition_average_list), "ids")

gas_nutrition_categories_list: 1135 ids
gas_average: 1152 ids
nutrition_average: 1137 ids


In [21]:
gas_clean_complete = pd.read_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/gas_clean_average_subsets_1234561h_2026_09_26.csv')
gas_clean_complete_list = gas_clean_complete["id_lab"].unique().tolist()
print('gas_clean_complete:', len(gas_clean_complete_list), "ids")

nutrition_complete = pd.read_csv('/content/drive/MyDrive/lmf/output/2026_09_26_trial_database_curation/nutrition_average_1234561h_2026_09_26.csv')
nutrition_complete_list = nutrition_complete["id_lab"].unique().tolist()
print('nutrition_complete:', len(nutrition_complete_list), "ids")

subsets_gas_sorted_by_sql = pd.read_csv('/content/drive/MyDrive/lmf/output/2026_09_24_sql_data_frame_creation_with_curated_subsets/subsets_1_2_3_4_5_6_1h_gas_sorted_by_sql.csv')
subsets_gas_sorted_by_sql_list = subsets_gas_sorted_by_sql["id_lab"].unique().tolist()
print('subsets_gas_sorted_by_sql:', len(subsets_gas_sorted_by_sql_list), "ids")

subsets_nutrition_sorted_by_sql = pd.read_csv('/content/drive/MyDrive/lmf/output/2026_09_24_sql_data_frame_creation_with_curated_subsets/subsets_1_2_3_4_5_6_1h_nutrition_sorted_by_sql.csv')
subsets_nutrition_sorted_by_sql_list = subsets_nutrition_sorted_by_sql["id_lab"].unique().tolist()
print('subsets_nutrition_sorted_by_sql:', len(subsets_nutrition_sorted_by_sql_list), "ids")

gas_clean_complete: 1224 ids
nutrition_complete: 1211 ids
subsets_gas_sorted_by_sql: 1248 ids
subsets_nutrition_sorted_by_sql: 1215 ids


In [22]:
difference_gas = set(subsets_gas_sorted_by_sql_list) - set(gas_clean_complete_list)
print('difference_gas:', len(difference_gas), "ids")
print(difference_gas)

difference_gas: 24 ids
{'F24-3545', 'F25-2131', 'F25-0850', 'F25-0003', 'F25-0773', 'F26-0571', 'F25-0770', 'F26-0569', 'F26-0087', 'F25-0769', nan, 'F26-0570', 'F25-1772', 'F25-0004', 'F26-0042', 'F25-2076', 'F24-3469', 'F25-0005', 'F25-0001', 'F25-0019', 'F25-2122', 'F25-0002', 'F25-1692', 'F25-0007'}


In [23]:
difference_nutrition = set(subsets_nutrition_sorted_by_sql_list) - set(nutrition_complete_list)
print('difference_nutrition:', len(difference_nutrition), "ids")
print(difference_nutrition)

difference_nutrition: 4 ids
{'F25-2122', 'F24-3469', 'F25-2076', 'F25-2131'}


In [24]:
absent = subsets_gas_sorted_by_sql[subsets_gas_sorted_by_sql["id_lab"].isin(difference_gas)]

In [25]:
absent.to_csv('absent.csv', index=False)